## 1. Importación de librerías

Framework: Pytorch

In [ ]:
!pip install opencv-python
!pip install -q torchinfo
!pip install -q ultralytics
#!pip install -q torch torchvision torchinfo tensorboard scikit-learn seaborn matplotlib numpy pandas opencv-python pillow h5py scipy tqdm

In [2]:
# Librerías de Python
import os
import json
import sys
import csv
import time
import random
import warnings
from glob import glob
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

# Manejo de datos y matemáticas
import numpy as np
import pandas as pd
import h5py

# Procesamiento de imágenes
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter

# Visualización y barras de progreso
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

# Machine Learning (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Deep Learning: PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

# Deep Learning: Torchvision (Computer Vision)
import torchvision
from torchvision import datasets
import torchvision.transforms as T
from torchvision.transforms import v2 as transforms
from torchvision.ops import Conv2dNormActivation

from collections import defaultdict

%matplotlib inline
warnings.filterwarnings("ignore")

In [3]:
VARIANTES_YOLO = {
    "yolov8n": dict(width=0.25, depth=0.33, max_ch=1024, pt="yolov8n.pt"),
    "yolov8s": dict(width=0.50, depth=0.33, max_ch=1024, pt="yolov8s.pt"),
    "yolov8m": dict(width=0.75, depth=0.67, max_ch=768,  pt="yolov8m.pt"),
    "yolov8l": dict(width=1.00, depth=1.00, max_ch=512,  pt="yolov8l.pt"),
    "yolov8x": dict(width=1.25, depth=1.00, max_ch=512,  pt="yolov8x.pt"),
}

Importación de funciones y clases ya definidas en el código de entrenamiento

In [ ]:
# En este caso, como se empleó Google Colab, se monta el drive para acceder a los archivos del proyecto
# En otro entorno, se debería ajustar la ruta de importación del código de entrenamiento

from google.colab import drive
#Para importar del código de entrenamiento En Google Colab
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

#from train_YOLO_crowdhuman import (
from train_YOLO_crowdhuman import ( #En Google Colab
    YOLO_model,
    TrainingConfig,
    cargar_anotaciones_cabezas,
    resize_image,
    xywh_to_xyxy,
    decode_to_xywh
)

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Descarga del dataset

3001 imágenes anotadas

| datos | nº imágenes | % |
| --- | --- | --- |
| training y validation | 2552 | 85,04% |
| test | 449 | 14,96% |

Estructura de directorios del entorno

In [ ]:
%%bash
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/IQBwS9aoohFwTLTjjUEdtw7gAYCxn3HUX7rWw0PaKVVNW8M?e=Ny6uQm&download=1" -O "crowdhuman.zip"
mkdir -p data/crowdhuman
unzip -q "crowdhuman.zip" -d data/crowdhuman

echo "Test:" $(ls data/crowdhuman/test/images | wc -l) "imágenes"

In [6]:
test_images_root = os.path.join("data", "crowdhuman", "test", "images")
test_root = os.path.join("data", "crowdhuman", "test")

#Comprobamos dimensiones de alguna de las imágenes (p.ej. 10 primeras)
for i, file in enumerate(sorted(os.listdir(test_images_root))[:10]):
    path = os.path.join(test_images_root, file)
    with Image.open(path) as img:
        print(f"{file}: {img.size}  (ancho x alto)")

273271-10de900086f10fd8_jpg.rf.276439811fb1e2b23ff8d847d6c60862.jpg: (800, 600)  (ancho x alto)
273271-10f400006b6fb935_jpg.rf.ca538e5dd1aa756c78229f9ba05b174b.jpg: (1024, 683)  (ancho x alto)
273271-112b80003f483eb4_jpg.rf.a641727e0e3a89433704b803ffcdb198.jpg: (1210, 907)  (ancho x alto)
273271-11b63000fdfc5d9d_jpg.rf.3cd2ee768930b6956d184873f7026e52.jpg: (1024, 768)  (ancho x alto)
273271-1236a000e7524e84_jpg.rf.91cd3e9d7bae0a01745c321a6faa47c4.jpg: (800, 533)  (ancho x alto)
273271-12822000b1291a69_jpg.rf.a358bedf3d3a920653ebc5a1fd0b0cd1.jpg: (1200, 756)  (ancho x alto)
273271-12d39000ac0f9be7_jpg.rf.6adb1a70ab9a76b284ec9e2b0760c8fb.jpg: (1024, 682)  (ancho x alto)
273271-14e9a00044d0e812_jpg.rf.1ca6216ac625bae7add6a60d632faa6b.jpg: (1100, 768)  (ancho x alto)
273271-15eec000cad7fba9_jpg.rf.4948adc687f043f7dd9e395242494686.jpg: (1000, 667)  (ancho x alto)
273271-165ab00051493658_jpg.rf.4ce4e0b32873b3f3f41357957df5603c.jpg: (720, 540)  (ancho x alto)


Carga de las anotaciones desde archivos de texto con el formato YOLO

In [7]:
ann = cargar_anotaciones_cabezas(test_root)
#Comprobación de las anotaciones
for idx, (img_id, labels) in enumerate(ann.items()):
        if idx == 10:  #Ver solo los idx primeros
          break
        labels_str = ",  ".join(f"clase: {clase} > ({cx},{cy},{bbox_w},{bbox_h})" for clase, cx, cy, bbox_w, bbox_h in labels)
        print(f"[{img_id}] -> [{labels_str}]")

[273275-5f94000988d0dee_jpg.rf.3a9f8d588fb7788ec1e3132e1eef09fa.jpg] -> [clase: 1 > (0.04662698412698413,0.5688244047619048,0.037698412698412696,0.03249007936507937),  clase: 1 > (0.2261904761904762,0.5788690476190477,0.036375661375661374,0.03571428571428571),  clase: 1 > (0.21841931216931218,0.5598958333333334,0.02017195767195767,0.018105158730158732),  clase: 1 > (0.2705026455026455,0.5530753968253969,0.023809523809523808,0.022817460317460316),  clase: 1 > (0.2878637566137566,0.5493551587301587,0.016203703703703703,0.013888888888888888),  clase: 1 > (0.304728835978836,0.5388144841269841,0.008267195767195767,0.006696428571428571),  clase: 1 > (0.3237433862433862,0.5465029761904762,0.006613756613756613,0.006200396825396825),  clase: 1 > (0.3229166666666667,0.541046626984127,0.00496031746031746,0.004712301587301587),  clase: 1 > (0.33630952380952384,0.5385664682539683,0.005291005291005291,0.004712301587301587),  clase: 1 > (0.36507936507936506,0.5394345238095238,0.009259259259259259,0.0

**Reproducibilidad experimental**

In [8]:
#Establecer semilla para que las ejecuciones sean reproducibles
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.cuda.manual_seed_all(seed)
       torch.backends.cudnn.deterministic = True
       torch.backends.cudnn.benchmark = False

Funciones auxiliares

In [9]:
def yolo_norm_to_resized_xyxy(gt_boxes_norm, r, pad_left, pad_top, w0, h0):
    """
    Convierte el GT en formato YOLO normalizado respecto a imagen original (w0,h0)
    a boxes XYXY en píxeles sobre la imagen resized (img_size)
    """
    #Si la lista está vacía, devolver tensor vacío
    if not gt_boxes_norm:
        return torch.zeros((0, 4))

    #Convertir la lista de listas de Python a un tensor de PyTorch
    targets = torch.tensor(gt_boxes_norm, dtype=torch.float32)

    # gt_boxes_norm es del tipo [clase, x, y, w, h], nos quedamos con las últimas 4
    boxes_norm = targets[:, 1:]
    device = boxes_norm.device

    #Pasar de [0, 1] al tamaño original de imagen, se multiplica cx y w por w0, y cy y h por h0
    scale_tensor = torch.tensor([w0, h0, w0, h0], device=device)
    boxes_pixels = boxes_norm * scale_tensor

    #Transformar de centro (xywh) a esquinas (xyxy)
    boxes_xyxy = xywh_to_xyxy(boxes_pixels)

    #Aplicar escalado y padding: (coord_nueva = (coord_orig * r) + pad)
    boxes_xyxy *= r
    pad_tensor = torch.tensor([pad_left, pad_top, pad_left, pad_top], device=device)
    boxes_xyxy += pad_tensor

    return boxes_xyxy

**Funciones de cálculo de métricas de rendimiento**

In [10]:
def match_tp_fp_fn(pred_boxes, pred_scores, gt_boxes_xyxy, iou_match=0.45):
    """
    Compara las predicciones del modelo con el Ground Truth usando Greedy Matching (mayor score).
    El objetivo es decidir para cada box si es un acierto (tp) o un falso positivo, utilizando IoU
    y asegurando que cada objeto real sea detectado una única vez por la predicción con mayor score.
    Args:
      - pred_boxes: Tensor [N,4] xyxy
      - pred_scores: Tensor [N]
      - gt_boxes_xyxy: Tensor [M,4] xyxy
      - iou_match: límite IoU
    Devuelve:
      - tp: aciertos
      - fp: falsos positivos
      - fn: falsos negativos
    """
    if pred_boxes is None or len(pred_boxes) == 0:
        n_gt = len(gt_boxes_xyxy) if gt_boxes_xyxy is not None else 0
        return 0, 0, n_gt

    pred_boxes   = pred_boxes.cpu()
    pred_scores  = pred_scores.cpu()
    gt_boxes_xyxy = gt_boxes_xyxy.cpu() if isinstance(gt_boxes_xyxy, torch.Tensor) else torch.tensor(gt_boxes_xyxy, dtype=torch.float32)

    if len(gt_boxes_xyxy) == 0:
        return 0, len(pred_boxes), 0

    #Matrizd de IoU [N, M]
    iou_matrix = torchvision.ops.box_iou(pred_boxes, gt_boxes_xyxy).numpy()
    #Ordenar predicciones por score descendente
    order = np.argsort(-pred_scores.numpy())
    iou_matrix = iou_matrix[order]

    matched_gt = np.zeros(iou_matrix.shape[1], dtype=bool)
    tp = fp = 0

    #Matching Voraz (Greedy Matching), iteramos sobre las predicciones (filas de la matriz)
    for row in iou_matrix:
        #Se pone -1 a los GT ya asignados para que no se elijan
        row = row.copy()
        row[matched_gt] = -1.0

        #Mejor GT disponible
        best_j = int(np.argmax(row))
        if row[best_j] >= iou_match:
            tp += 1
            matched_gt[best_j] = True
        else:
            fp += 1

    fn = int(iou_matrix.shape[1]) - tp
    return tp, fp, fn

def precompute_iou_cache(model_outputs_by_image, gt_by_image, score_threshold=0.01):
    """
    Calcula la matriz IoU [N, M] por imagen

    Args:
      - model_outputs_by_image: dict img -> (boxes_all, scores_all)
      - gt_by_image: dict img -> list gt boxes xyxy
      - score_threshold: umbral de score para filtrar predicciones

    Devuelve iou_cache: dict  img_name -> dict con:
      'pred_scores' : np.ndarray [N]      scores filtrados por score_threshold
      'pred_indices': lista de índices originales
      'iou_mat'     : np.ndarray [N, M]   IoU entre cada pred y cada GT
      'n_gt'        : int                 número de GT en la imagen
    Las imágenes sin predicciones o sin GT iou_mat = None
    """
    iou_cache = {}

    #Para cada imagen del conjunto test
    for img_name, gt_boxes in gt_by_image.items():
        # GT a CPU numpy
        if isinstance(gt_boxes, torch.Tensor):
            gt_cpu = gt_boxes.cpu()
        else:
            gt_cpu = torch.tensor(gt_boxes, dtype=torch.float32)

        n_gt = len(gt_cpu)

        boxes, scores = model_outputs_by_image.get(img_name, (None, None))

        #Imágenes sin predicción
        if boxes is None or len(boxes) == 0 or n_gt == 0:
            iou_cache[img_name] = {
                'pred_scores': np.array([]),
                'iou_mat': None,
                'n_gt': n_gt
            }
            continue

        #Predicciones a CPU numpy
        boxes_cpu  = boxes.cpu()  if isinstance(boxes,  torch.Tensor) else torch.tensor(boxes)
        scores_cpu = scores.cpu() if isinstance(scores, torch.Tensor) else torch.tensor(scores)

        #Filtrar predicciones con score muy bajo (descartes)
        mask = scores_cpu.numpy() >= score_threshold
        boxes_cpu  = boxes_cpu[mask]
        scores_cpu = scores_cpu[mask]

        if len(boxes_cpu) == 0:
            iou_cache[img_name] = {
                'pred_scores': np.array([]),
                'iou_mat': None,
                'n_gt': n_gt
            }
            continue

        #LLamada al cálculo de IoU de torchvision por imagen
        iou_mat = torchvision.ops.box_iou(boxes_cpu, gt_cpu).numpy()  # [N, M]

        iou_cache[img_name] = {
            'pred_scores': scores_cpu.numpy(),  # [N]
            'iou_mat': iou_mat,                 # [N, M]
            'n_gt': n_gt
        }

    return iou_cache

def compute_ap_for_iou(iou_cache, iou_match):
    """
    Calcula el Average Precision (AP) para un IoU (iou_match).
    Utiliza los arrays de precisión y recall para calcular la integral y medir el
    Área Bajo la Curva (AUC) de precisión-recall con la función trapz.

    Args:
      - iou_cache: salida de precompute_iou_cache()
      - iou_match: umbral de IoU
    Devuelve:
      - AP al iou_match dado
    """

    #Juntar todas las predicciones en una lista global como (img_name, score, pred_idx)
    all_preds = []
    total_gt = 0

    for img_name, cache in iou_cache.items():
        total_gt += cache['n_gt']
        scores = cache['pred_scores']
        if len(scores) == 0:
            continue
        for k, s in enumerate(scores):
            all_preds.append((img_name, float(s), k))

    if total_gt == 0:
        return 0.0

    #Ordenar por score descendente
    all_preds.sort(key=lambda x: x[1], reverse=True)

    #Seguimiento de GTs ya asignados por imagen
    matched = {img: np.zeros(cache['n_gt'], dtype=bool)
               for img, cache in iou_cache.items()}

    tp_list, fp_list = [], []

    #Bucle de evaluación
    for img_name, score, pred_idx in all_preds:
        cache = iou_cache[img_name]
        iou_mat = cache['iou_mat']

        #Si no hay GT en la imagen es FP directamente
        if iou_mat is None or cache['n_gt'] == 0:
            tp_list.append(0); fp_list.append(1)
            continue

        #Fila de IoUs para esta predicción
        row = iou_mat[pred_idx].copy()
        row[matched[img_name]] = -1.0  #enmascarar GTs ya asignados

        best_j   = int(np.argmax(row))
        best_iou = row[best_j]

        #garantiza que aunque dos predicciones apunten al mismo objeto real, solo la de mayor score se lleva el TP
        if best_iou >= iou_match and not matched[img_name][best_j]:
            tp_list.append(1); fp_list.append(0)
            matched[img_name][best_j] = True
        else:
            tp_list.append(0); fp_list.append(1)

    #Calcular métricas acumuladas
    tp_acum    = np.cumsum(tp_list)
    fp_acum    = np.cumsum(fp_list)
    recalls    = tp_acum / (total_gt + 1e-6)
    precisions = tp_acum / (tp_acum + fp_acum + 1e-6)

    return float(np.trapz(precisions, recalls))

def compute_map(model_outputs_by_image, gt_by_image, score_threshold=0.01, verbose=True):
    """
    Calcula AP@0.50 y mAP@0.50:0.95
      Precalcula matrices IoU una sola vez
      Iterar sobre 10 umbrales IoU reutilizando la caché

    Args:
      - model_outputs_by_image: dict  img_name -> (boxes_tensor, scores_tensor)
      - gt_by_image: dict  img_name -> tensor GT xyxy
      - score_threshold: umbral de score para filtrar predicciones

    Devuelve:
      - ap50
      - map5095
    """
    if verbose:
        print("[AP] Precálculo de matrices IoU por imagen...")

    iou_cache = precompute_iou_cache(
        model_outputs_by_image, gt_by_image, score_threshold=score_threshold
    )

    #AP@0.50
    ap50 = compute_ap_for_iou(iou_cache, iou_match=0.45)
    if verbose:
        print(f"[AP] AP@0.50 = {ap50:.4f}")

    #mAP@0.50:0.95 (10 umbrales), reutilizando caché
    iou_list = list(np.arange(0.50, 0.96, 0.05))
    aps = []
    iterator = tqdm(iou_list, desc="mAP@50:95") if verbose else iou_list
    for iou in iterator:
        aps.append(compute_ap_for_iou(iou_cache, iou_match=float(iou)))

    map5095 = float(np.mean(aps))
    if verbose:
        print(f"[AP] mAP@0.50:0.95 = {map5095:.4f}")

    return ap50, map5095

**Función de Post-Procesamiento**

In [11]:
def decode_predictions(prediction, img_size=640, iou_nms=0.65):
    """
    Decodifica predicciones y aplica NMS
    Devuelve:
        boxes: Tensor [N,4] xyxy (píxeles)
        scores: Tensor [N]
    """
    strides = [8, 16, 32]
    all_boxes = []
    all_scores = []
    device = prediction[0].device

    for i, pred in enumerate(prediction):
        stride = strides[i]
        B, C, H, W = pred.shape

        #[B, C, H, W] -> [B, H, W, C]
        pred = pred.permute(0, 2, 3, 1).contiguous()

        if C < 6:
            raise ValueError(f"Canales insuficientes. Se esperan >=6 (box+obj+cls). C={C}")

        box_raw = pred[..., :4]
        obj = pred[..., 4].sigmoid()
        cls = pred[..., 5].sigmoid() # (nc=1)

        scores = (obj * cls).reshape(-1)

        grid_y, grid_x = torch.meshgrid(
            torch.arange(H, device=device),
            torch.arange(W, device=device),
            indexing='ij'
        )
        grid = torch.stack([grid_x, grid_y], dim=-1).float() # [H, W, 2]

        boxes_norm = decode_to_xywh(box_raw, grid.unsqueeze(0), stride, img_size, device)
        boxes_pixel = boxes_norm * img_size
        boxes_xyxy = xywh_to_xyxy(boxes_pixel)

        # [B, H, W, 4] -> [N, 4]
        all_boxes.append(boxes_xyxy.reshape(-1, 4))
        all_scores.append(scores)

    if len(all_boxes) == 0:
        return None, None

    cat_boxes = torch.cat(all_boxes, dim=0)
    cat_scores = torch.cat(all_scores, dim=0)

    keep = torchvision.ops.nms(cat_boxes, cat_scores, iou_nms)
    return cat_boxes[keep], cat_scores[keep]


## Función principal de test

In [12]:
def test_yolo(
    model,
    weights_path,
    images_dir,
    output_dir,
    img_size=640,
    iou_nms=0.65,
    iou_match=0.45,
    thresholds=np.linspace(0.05, 0.95, 19),
    seed=28,
    device=None,
    annotations=None,
    save_visuals=False,
    results_csv="results.csv",
    summary_csv="summary.csv"
):

    """
    Test y evaluación de un modelo YOLO

    - model: modelo YOLO cargado
    - weights_path: pesos entrenados
    - images_dir: carpeta de imágenes
    - output_dir: carpeta de resultados
    - img_size: tamaño de entrada YOLO
    - iou_nms: límite IoU para NMS
    - device: 'cuda' o 'cpu'
    - annotations: GT en formato YOLO
    - results_csv: CSV por imagen
    - summary_csv: CSV resumen métricas
    - save_visuals: guardar imágenes con las detecciones
    """

    set_seed(seed)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    os.makedirs(output_dir, exist_ok=True)
    if save_visuals:
        vis_dir = os.path.join(output_dir, "visuals")
        os.makedirs(vis_dir, exist_ok=True)


    #Cargar los pesos entrenados
    print(f"[INFO] Cargando pesos desde {weights_path}...")
    state = torch.load(weights_path, map_location=device)
    try:
        model.load_state_dict(state)
    except RuntimeError:
        print("[WARN] Falló la carga estricta, intentando con strict=False...")
        model.load_state_dict(state, strict=False)

    img_paths = sorted(
        p for p in Path(images_dir).glob("*")
        if p.suffix.lower() in [".jpg", ".png", ".jpeg"]
    )

    if save_visuals:
        random.seed(28)
        #Seleccionar 10 imágenes al azar para visualización
        vis_samples = set(random.sample(img_paths, k=min(10, len(img_paths))))

    results_count = []
    mae_errors, rmse_errors = [], []
    inference_times = []
    raw_rows = []

    #Diccionarios para el cálculo final de mAP
    if "outputs_by_image" not in locals():
        outputs_by_image = {}
    if "gt_by_image" not in locals():
        gt_by_image = {}

    with torch.no_grad():
        for img_path in tqdm(img_paths, desc="Testing YOLO11"):
            img = Image.open(img_path).convert("RGB")
            img_lb, r, pad_left, pad_top, new_w, new_h, w0, h0 = resize_image(img, img_size)
            img_t = T.ToTensor()(img_lb).unsqueeze(0).to(device)

            #Inferencia
            start = time.time()
            preds = model(img_t)
            if device == "cuda":
                torch.cuda.synchronize()
            inference_times.append(time.time() - start)

            #Decodificaión
            boxes_all, scores_all = decode_predictions(preds, img_size=img_size, iou_nms=iou_nms)

            #Procesar Ground Truth
            gt_boxes_norm = annotations.get(img_path.name, [])
            gt_count = len(gt_boxes_norm)

            gt_xyxy_lb = torch.empty((0, 4), device=device)
            if gt_count > 0:
                gt_xyxy_lb = yolo_norm_to_resized_xyxy(gt_boxes_norm, r, pad_left, pad_top, w0, h0)

            #Guardar para mAP global
            outputs_by_image[img_path.name] = (boxes_all, scores_all)
            gt_by_image[img_path.name] = gt_xyxy_lb

            #Conteo (siempre se usa por consistencia un conf_thres para conteo fijo, p.ej. 0.25)
            # Nota: tu conteo es un proxy, pero lo dejamos para no romper tu flujo
            conf_for_count = 0.25
            pred_count = 0
            if boxes_all is not None and scores_all is not None:
                pred_count = int((scores_all >= conf_for_count).sum().item())

            error = pred_count - gt_count
            mae_errors.append(abs(error))
            rmse_errors.append(error ** 2)

            results_count.append({
                "image": img_path.name,
                "gt_count": gt_count,
                "pred_count": pred_count,
                "error": error,
                "abs_error": abs(error)
            })

            #Sweep thresholds
            for th in thresholds:
                if boxes_all is None:
                    boxes_th = torch.empty((0,4), device=device)
                    scores_th = torch.empty((0,), device=device)
                else:
                    m = scores_all >= float(th)
                    boxes_th = boxes_all[m]
                    scores_th = scores_all[m]
                    pred_count = int(m.sum().item()) # Corrected 'mask' to 'm'

                error = pred_count - gt_count
                abs_error = abs(error)
                sq_error = error ** 2

                tp, fp, fn = match_tp_fp_fn(
                    pred_boxes=boxes_th,
                    pred_scores=scores_th,
                    gt_boxes_xyxy=gt_xyxy_lb,
                    iou_match=iou_match
                )

                raw_rows.append({
                    "image": img_path.name,
                    "threshold": float(th),
                    "tp": int(tp),
                    "fp": int(fp),
                    "fn": int(fn),
                    "pred_count": pred_count,
                    "gt_count": gt_count,
                    "abs_error": abs_error,
                    "sq_error": sq_error
                })

            #Visualización (usa un threshold fijo)
            if save_visuals and img_path in vis_samples:
                save_path = os.path.join(vis_dir, f"{img_path.stem}.png")
                img_vis_np = np.array(img_lb)

                #Para visualizar se filtra con conf_for_count
                if boxes_all is None:
                    boxes_vis = None
                else:
                    boxes_vis = boxes_all[scores_all >= conf_for_count]


                ###### PLOT simple para la visualización de las predicciones
                img_pred = img_vis_np.copy()
                if boxes_vis is not None:
                    for box in boxes_vis:
                        x1, y1, x2, y2 = box.int().cpu().numpy()
                        cv2.rectangle(img_pred, (x1, y1), (x2, y2), (255, 0, 0), 2)

                fig, ax = plt.subplots(1, 1, figsize=(6, 6))
                ax.imshow(img_pred)
                ax.set_title(f"GT: {gt_count} | Pred@{conf_for_count}: {0 if boxes_vis is None else len(boxes_vis)}")
                ax.axis("off")
                plt.tight_layout()
                plt.savefig(save_path)
                plt.close(fig)

    ###### Resumen detección por threshold #######
    df_raw = pd.DataFrame(raw_rows)

    #Agrupamos por threshold y calculamos sumas (TP, FP, FN) y medias (para calcular MAE y MSE global)
    df_sum = df_raw.groupby("threshold").agg({
        "tp": "sum",
        "fp": "sum",
        "fn": "sum",
        "abs_error": "mean",  #MAE por threshold
        "sq_error": "mean"    #MSE
    }).reset_index()

    df_sum["precision"] = df_sum["tp"] / (df_sum["tp"] + df_sum["fp"] + 1e-6)
    df_sum["recall"] = df_sum["tp"] / (df_sum["tp"] + df_sum["fn"] + 1e-6)
    df_sum["f1"] = 2 * df_sum["precision"] * df_sum["recall"] / (df_sum["precision"] + df_sum["recall"] + 1e-6)

    #Calcular métricas de conteo
    df_sum["MAE"] = df_sum["abs_error"]
    df_sum["RMSE"] = np.sqrt(df_sum["sq_error"])

    #Selección del mejor punto de operación (Best F1)
    best_idx = df_sum["f1"].idxmax()
    best_row = df_sum.loc[best_idx]

    ###### Resumen de conteo #######
    summary = {
        "Best_Threshold": float(best_row["threshold"]),
        "Best_F1": float(best_row["f1"]),
        "Best_Precision": float(best_row["precision"]),
        "Best_Recall": float(best_row["recall"]),
        "MAE_count": float(np.mean(mae_errors)),
        "RMSE_count": float(np.sqrt(np.mean(rmse_errors))),
        "mean_inference_time": float(np.mean(inference_times)),
    }

    #Guardar csv
    if results_csv:
        pd.DataFrame(results_count).to_csv(os.path.join(output_dir, results_csv), index=False)

    #df_raw.to_csv(os.path.join(output_dir, "raw_by_image_threshold.csv"), index=False)
    df_sum.to_csv(os.path.join(output_dir, "metrics_by_threshold.csv"), index=False)

    ###### Cálculo de mAP ######
    ap50, map5095 = compute_map(outputs_by_image, gt_by_image)

    summary["AP@0.50"] = ap50
    summary["AP@0.50:0.95"] = map5095

    if summary_csv:
        pd.DataFrame([summary]).to_csv(os.path.join(output_dir, summary_csv), index=False)

    return results_count, summary, df_sum, df_raw

## Main

In [14]:
import sys

seed=94
variante="yolov8s"

train_config = TrainingConfig(variante=variante)

model = YOLO_model(
    num_classes=1,
    width=train_config.width,
    depth=train_config.depth,
    max_ch=train_config.max_ch
)

results, summary, df_sum, df_raw = test_yolo(
    model=model,
    weights_path="yolo_nano_best.pth",
    images_dir=test_images_root,
    annotations=ann,
    img_size=640,
    iou_nms=0.45,
    iou_match=0.45,
    seed=seed,
    output_dir="output/test_results",
    save_visuals=False,
    results_csv=f"results_{variante}_seed{seed}.csv",
    summary_csv=f"summary_{variante}_seed{seed}.csv",
)

print(summary)

[INFO] Cargando pesos desde yolo_nano_best.pth...


Testing YOLO11: 100%|██████████| 449/449 [00:46<00:00,  9.76it/s]


[AP] Precálculo de matrices IoU por imagen...
[AP] AP@0.50 = 0.5372


mAP@50:95: 100%|██████████| 10/10 [01:37<00:00,  9.74s/it]

[AP] mAP@0.50:0.95 = 0.2012
{'Best_Threshold': 0.3, 'Best_F1': 0.585683038086952, 'Best_Precision': 0.6200046739168025, 'Best_Recall': 0.5549628699346342, 'MAE_count': 7.195991091314031, 'RMSE_count': 14.556318705424719, 'mean_inference_time': 0.019206877540639353, 'AP@0.50': 0.5372267158840563, 'AP@0.50:0.95': 0.20124466141182693}
